# Creation of xyz/cif files for different sizes and crystals forms from a dataset of compounds (cif files)

## Imports

In [1]:
import os
import sys
import tkinter

print(os.getcwd())
cwd0 = './styles/'
sys.path.append(cwd0)
from pathlib import Path
import re
import numpy as np
import pandas as pd
import visualID as vID
from visualID import  fg, hl, bg

from pyNanoMatBuilder import crystalNPs as cyNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib

from ase import io
from ase.atoms import Atoms
from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup
from ase.io import read
from ase.io import write
from ase.visualize import view

from debyecalculator import DebyeCalculator
import pyNanoMatBuilder.utilsDC as pyNMBuDC

/home/sara/pyNanoMatBuilder


## Create xyz/cif files for the predefined Wulff forms

### Definition of the PredifinedWulffFiles class : functions that load cif files, extract the data, and call the crystalNPs class to create the Wulff nps if the lattice of the compounds matches with the Wulff form

Two functions are defined in utils : `load_cif` and `writexyz_generalized_crystals`

##### 1) code for a dataset of cif files

In [2]:

class PredifinedWulffFiles :
    """
    A class for generating XYZ and CIF files of Predefined Wulff forms nanoparticles (NPs) 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data, wulff_shapes,sizes,form: str=None, max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data, Wulff shapes information and size.
        Args:
            path (str) : Path where the created xyz files will be stored.
            cif data (Dataframe):  DataFrame containing CIF files of different compounds.
            wulff_shapes (Dataframe): Dataframe of the wulff shapes and their informations
            sizes (array-like): Array of the multiplier coefficients of dhkl, ex: [[2],[3]] will give the final size [[2*dhkl],[3*dhkl]]
            form (str, optional): if str=None : If None, all Wulff forms are created; otherwise, a specific form is selected.
            max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
            noOutput (bool): if bool=False : details of the files
        Methods:
        create_wulff_shapes(noOutput, max_size, path):  Generate Wulff shapes and their files for all the CIF compounds dataset.
        """
        self.cif_data = cif_data  # DataFrame containing CIF data
        self.wulff_shapes = wulff_shapes  # DataFrame of Wulff forms
        self.loaded_cifs = {}  # Stock loaded cif files
        self.sizes= [[k] for k in sizes]#  List of sizes (nested list of the sizes) 
        self.form= form # Specific form to generate (if any)
        
        self.create_wulff_shapes(noOutput,max_size,path)  # Call method to generate Platonic nanoparticles
  

    def create_wulff_shapes(self,noOutput,max_size,path):
        """
        Generate Wulff shapes and their files for all the CIF compounds dataset.
        Args:
            noOutput (bool): If False, prints details about the process.
            max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
            path (str): Path where xyz/cif files will be created.
        """
        for cif_name, cif_file in self.cif_data['cif file'].items():
            
            # Extract the structure name for the name of the files, for example 'Rutile' or 'Anatase' or 'Alpha'
            self.cif_name=cif_name
            if len(self.cif_name.split())==2 : # For the name of the files, for example 'Rutile' or 'Anatase'
                structure=self.cif_name.split()[1]
            else : 
                structure=None
            if not noOutput :
                print(f'\n\033[1m {bg.LIGHTBLUEB} {cif_name.center(50)}\033[0m\n') 
            cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
            crystal_system_name = self.ucBL.__class__.__name__
            direction=[0,0,1]
            d_hkl=pyNMBu.interPlanarSpacing(direction,self.ucUnitcell,crystal_system_name)*0.1 #nm    
            if not noOutput:
                print(f'\033[1m d_hkl={d_hkl} nm \033[0m')
            
            if self.form == None : # If no specific form is selected, generate all possible forms
                for form, row in self.wulff_shapes.iterrows():
                    lattices = [l.strip() for l in row['Bravais lattice'].split(',')]
                    if self.crystal_type  in lattices:  # Verify if the lattice of the compound matches the lattice of the wulff form
                        index=0 
                        if not noOutput :
                            print(f"\n {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {lattices} of the Wulff {form} form. \033[0m \n")
                        
                        # Create instances for each form and size
                        for i in self.sizes :
                            size= [i[0]*d_hkl]
                            index += 1 
                            TestNP = cyNP.Crystal(
                                crystal=f'{cif_name}',
                                userDefCif=cif_info['cif_path'],
                                shape=f"Wulff: {form}",
                                sizesWulff= size,
                                threshold=0.001,
                                thresholdCoreSurface=2,
                                postAnalyzis=True,
                                jmolCrystalShape=True,
                                noOutput=True,
                                aseView=False,
                                skipSymmetryAnalyzis=True
                            ) 
                            circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                            if circumsphere_diameter<max_size :
                                if not noOutput :
                                    print(f'\033[1m Generating size is {size[0]:.4f} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                                    print(f'\033[1m  Circumscribed sphere diameter ={circumsphere_diameter} nm \033[0m')
                                pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)     
                            else :
                                if not noOutput :
                                    print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm \033[0m') 
                                
                                break
                    else:
                        if not noOutput :
                            print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {lattices} of the Wulff {form} form .\033[0m  ")     
                            
           
            else:  # If a specific form is given by the user
                   
                lattices = self.wulff_shapes['Bravais lattice'][f'{self.form}']
                if self.crystal_type  == lattices:
                    index=0 
                    if not noOutput :
                        print(f"\n {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {lattices} of the Wulff {self.form} form.\033[0m\n ")
                    # instance of the crystal class for each cif file and possible form and size
                    for i in self.sizes :
                        size= [i[0]*d_hkl]
                        if not noOutput :
                            print(f'\033[1m Generating size is {size[0]:.4f} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                        index += 1 
                        TestNP = cyNP.Crystal(
                            crystal=f'{cif_name}',
                            userDefCif=cif_info['cif_path'],
                            shape=f"Wulff: {self.form}",
                            sizesWulff= size,
                            threshold=0.001,
                            thresholdCoreSurface=2,
                            postAnalyzis=True,
                            jmolCrystalShape=True,
                            noOutput=True,
                            aseView=False,
                            skipSymmetryAnalyzis=True
                        ) 
                        circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                        if circumsphere_diameter<max_size :
                            if not noOutput :
                                print(f'\033[1m  Generating size is {size[0]:.4f} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                                print(f'\033[1m  Circumscribed sphere diameter ={circumsphere_diameter} nm\033[0m')
                            pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)     
                        else :
                            if not noOutput :
                                print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm. \033[0m')
                            break
             
                else:
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {lattices} of the Wulff {self.form} form.\033[0m ")    

      

##### 2) Code for a single cif file

In [3]:
class PredifinedWulffFiles_onecif :
    """
    A class for generating XYZ and CIF files of Predefined Wulff forms nanoparticles (NPs) 
    from a single specific CIF compound. 
    """
    def __init__(self, path, cif_file, cif_name, wulff_shapes,sizes,form: str=None,max_size: float=50, noOutput:bool = True):
        """
        Initialize the class with CIF data, Wulff shapes information and size.
            Args:
        path (str) : path that will contain the created CIF/xyz files
        cif file (file): singular cif file
        wulff_shapes (dataframe): the wulff shapes and their informations
        sizes (array-like): array of the sizes of the nanoparticles
        form (str): if str=None : all the wulff forms are created but possiblity to pick a specific form
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): if bool=False : details of the files
            Methods:
        create_wulff_shapes(cif_file,noOutput, path):  Generate Wulff shapes and their files for a single specific CIF coumpound.
        """
        self.cif_file = cif_file  # cif file
        self.wulff_shapes = wulff_shapes  # DataFrame of Wulff  forms
        self.loaded_cifs = {}  # stock loaded cif files
        self.sizes= [[k] for k in sizes] #nested list of the sizes 
        self.form= form # Specific form to generate (if any)
        self.cif_name= cif_name
        # number_created_files=0
        self.create_wulff_shapes(cif_file,noOutput,max_size,path)
  

    def create_wulff_shapes(self,cif_file,noOutput,max_size,path):
        """
        Generate Wulff shapes and their files for a single specific CIF coumpound.
         Args:
        cif file (file): singular cif file
        noOutput (bool): if bool=False : details of the files
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        path (str) : path that will contain the created xyz/CIF files
        
        """            
        cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
        if len(self.cif_name.split())==2 : # For the name of the files, for example 'Rutile' or 'Anatase'
                structure=self.cif_name.split()[1]
        else : 
            structure=None
        if not noOutput :
           print(f'\n\033[1m{self.cif_name.center(50)}\033[0m\n') 
            
        crystal_system_name = self.ucBL.__class__.__name__
        direction=[0,0,1]
        d_hkl=pyNMBu.interPlanarSpacing(direction,self.ucUnitcell,crystal_system_name)*0.1 #nm    
        if not noOutput:
            print(f'\033[1m d_hkl={d_hkl} nm \033[0m')
            
        if self.form == None : #if the form isn't specific : 
            for form, row in self.wulff_shapes.iterrows():
                lattices = [l.strip() for l in row['Bravais lattice'].split(',')]
                if self.crystal_type  in lattices:  #verify if the lattice of the cif structure matches the lattice of the wulff form
                    index=0 
                    if not noOutput :
                        print(f"\n {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {lattices} of the Wulff {form} form. \033[0m\n ")
                    # instance of the crystal class for each cif file and possible form and size
                    for i in self.sizes :
                        size= [i[0]*d_hkl]
                        index += 1 
                        TestNP = cyNP.Crystal(
                            crystal=f'{self.cif_name}',
                            userDefCif=cif_info['cif_path'],
                            shape=f"Wulff: {form}",
                            sizesWulff= size,
                            threshold=0.001,
                            thresholdCoreSurface=2,
                            postAnalyzis=True,
                            jmolCrystalShape=True,
                            noOutput=True,
                            aseView=False,
                            skipSymmetryAnalyzis=True
                        ) 
                        circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                        if circumsphere_diameter<max_size :
                            if not noOutput :
                                print(f'\033[1m Generating size is {size[0]:.4f} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                                print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm \033[0m')
                            pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)     
                        else :
                            if not noOutput :
                                print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm \033[0m')
                            break
                             
                else:
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {lattices} of the Wulff {form} form .\033[0m ")     
                        
       
        else:  # If a specific form is given by the user
               
            lattices = self.wulff_shapes['Bravais lattice'][f'{self.form}']
            if self.crystal_type in lattices:
                index=0 
                if not noOutput :
                    print(f"\n {bg.LIGHTGREENB} {self.crystal_type} corresponds to the lattices {lattices} of the Wulff {self.form} form.\033[0m\n ")
                # instance of the crystal class for each cif file and possible form and size
                for i in self.sizes :
                    size= [i[0]*d_hkl]
                    index += 1 
                    TestNP = cyNP.Crystal(
                        crystal=f'{self.cif_name}',
                        userDefCif=cif_info['cif_path'],
                        shape=f"Wulff: {self.form}",
                        sizesWulff= size,
                        threshold=0.001,
                        thresholdCoreSurface=2,
                        postAnalyzis=True,
                        jmolCrystalShape=True,
                        noOutput=True,
                        aseView=False,
                        skipSymmetryAnalyzis=True
                    ) 
                    circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                    if circumsphere_diameter<max_size :
                        if not noOutput :
                            print(f'\033[1m Generating size is {size[0]:.4f} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                            print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm \033[0m')
                        pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)     
                    else :
                        if not noOutput :
                            print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm \033[0m')
                        break
                    
     
            else:
                if not noOutput :
                    print(f" {bg.LIGHTREDB} {self.crystal_type} does not correspond to the lattices {lattices} of the Wulff {self.form} form .\033[0m ")    

      

#### Examples of use of the class PredifinedWulffFiles that creates the files

If noOutput=False :  
 - <span style="color:#008000"> in green : the allowed shapes created (the lattice matches with the shape) </span> 
 - <span style="color:#FF0000">in red : the shapes that are not created (the lattice doesn't match with the shape)</span>

##### 1) Create all the possible Wulff forms from a dataset of cif files

In [3]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/wulff_1_5nm' 

# Sizes of the NPs
size = np.arange(2, 30, 1) # d is a multiplier coefficient of dhkl, so the final sizes are : 2*dhkl, 3*dhkl, ..., 10*dhkl

# Instance of the class
# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the size given (by multiple of dhkl).
class_test=PredifinedWulffFiles(path,cif_data=data.pyNMBcif.CIFdf, wulff_shapes=data.WulffShapes.WSdf,sizes=size,max_size=5,noOutput= False) # Here a maximum of a 1 nm for the circumscribed sphere diameter.
print(t.chrono_stop(hdelay=True))


                         NaCl                       

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1000041-NaCl.cif
 d_hkl=0.562 nm 

  fcc corresponds to the lattices ['bcc', 'fcc', 'cubic'] of the Wulff cube form.  



/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


 Generating size is 1.1240 nm and is equal to dhkl multiplied by [2]. 
  Circumscribed sphere diameter =1.9468251077074183 nm 
xyz file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000001_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000001_0000000.cif
script file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000001_0000000.script
 Generating size is 1.6860 nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =2.9202376615611274 nm 
xyz file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000002_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000002_0000000.cif
script file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000002_0000000.script
 Generating size is 2.2480 nm and is equal to dhkl multiplied by [4]. 
  Circumscribed sphere diameter =3.893650215414837 nm 
xyz file created: database_1_5nm/wulff_1_5nm/NaCl_fcc_ cube_0000003_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/NaC

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'tetragonal' is not interpreted for space group Spacegroup(141, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


 Generating size is 1.1443 nm and is equal to dhkl multiplied by [4]. 
  Circumscribed sphere diameter =1.9820203801172178 nm 
xyz file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000003_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000003_0000000.cif
script file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000003_0000000.script
 Generating size is 1.4304 nm and is equal to dhkl multiplied by [5]. 
  Circumscribed sphere diameter =2.4775254751465225 nm 
xyz file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000004_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000004_0000000.cif
script file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000004_0000000.script
 Generating size is 1.7165 nm and is equal to dhkl multiplied by [6]. 
  Circumscribed sphere diameter =2.973030570175827 nm 
xyz file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000005_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/Fe_bcc_ cube_0000

KeyboardInterrupt: 

##### 2) Create all the possible Wulff forms from one cif file

In [13]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/wulff_1_5nm'

# Sizes of the NPs
sizes=np.arange(1,25,1)  # d is a multiplier coefficient of dhkl, so the final sizes are : 2*dhkl, 3*dhkl, ..., 10*dhkl

# Instance of the class
# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the size given (by multiple of dhkl).
class_test=PredifinedWulffFiles_onecif(path,cif_file='cod1539039-Fe_beta.cif',cif_name='Fe beta', wulff_shapes=data.WulffShapes.WSdf,sizes=sizes,max_size=5,noOutput= False) # Here a maximum of a 1 nm for the circumscribed sphere diameter.
print(t.chrono_stop(hdelay=True))

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1539039-Fe_beta.cif

                     Fe beta                      

 d_hkl=0.6345000000000001 nm 

  cubic corresponds to the lattices ['bcc', 'fcc', 'cubic'] of the Wulff cube form. 
 
 Generating size is 0.6345 nm and is equal to dhkl multiplied by [1]. 
 Circumscribed sphere diameter =0.8369600969681887 nm 
xyz file created: database_1_5nm/wulff_1_5nm/Fe_beta_ cube_0000001_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/Fe_beta_ cube_0000001_0000000.cif
script file created: database_1_5nm/wulff_1_5nm/Fe_beta_ cube_0000001_0000000.script
 Generating size is 1.2690 nm and is equal to dhkl multiplied by [2]. 
 Circumscribed sphere diameter =2.0638961538418052 nm 
xyz file created: database_1_5nm/wulff_1_5nm/Fe_beta_ cube_0000002_0000000.xyz
cif file created: database_1_5nm/wulff_1_5nm/Fe_beta_ cube_0000002_0000000.cif
script file created: database_1_5nm/wulff_1_5nm/Fe_beta_ cube_0000002_0000000.script
 Gene

##### 3) Create a specific Wulff form from a dataset of cif files (ex: Wires)

In [14]:
t = pyNMBu.timer()
t.chrono_start()

path='any_files_test'
sizes=np.arange(1,5,1)

class_test=PredifinedWulffFiles(path,cif_data=data.pyNMBcif.CIFdf, wulff_shapes=data.WulffShapes.WSdf,sizes=sizes,form='hcpsph1',noOutput=False)


print(t.chrono_stop(hdelay=True))


                         NaCl                       

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1000041-NaCl.cif
 d_hkl=0.562 nm 
  fcc does not correspond to the lattices hcp of the Wulff hcpsph1 form. 

                     TiO2 rutile                    

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9015662-TiO2-rutile.cif
 d_hkl=0.29587 nm 
  tetragonal does not correspond to the lattices hcp of the Wulff hcpsph1 form. 

                     TiO2 anatase                   

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9015929-TiO2-anatase.cif
 d_hkl=0.9514300000000001 nm 
  tetragonal does not correspond to the lattices hcp of the Wulff hcpsph1 form. 

                        Fe bcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod5000217-Fe_bcc.cif
 d_hkl=0.28608 nm 
  bcc does not correspond to the lattices hcp of the Wulff hcpsph1 form. 

                       Mn alpha    

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'tetragonal' is not interpreted for space group Spacegroup(141, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


  Generating size is 0.4069 nm and is equal to dhkl multiplied by [1]. 
  Circumscribed sphere diameter =0.35382473313136753 nm
xyz file created: any_files_test/Co_hcp_ hcpsph1_0000001_0000000.xyz
cif file created: any_files_test/Co_hcp_ hcpsph1_0000001_0000000.cif
script file created: any_files_test/Co_hcp_ hcpsph1_0000001_0000000.script
 Generating size is 0.8137 nm and is equal to dhkl multiplied by [2]. 
  Generating size is 0.8137 nm and is equal to dhkl multiplied by [2]. 
  Circumscribed sphere diameter =0.9793402630779383 nm
xyz file created: any_files_test/Co_hcp_ hcpsph1_0000002_0000000.xyz
cif file created: any_files_test/Co_hcp_ hcpsph1_0000002_0000000.cif
script file created: any_files_test/Co_hcp_ hcpsph1_0000002_0000000.script
 Generating size is 1.2206 nm and is equal to dhkl multiplied by [3]. 
  Generating size is 1.2206 nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =1.4574263693530343 nm
xyz file created: any_files_test/Co_hcp_ hcpsph1_0

## Create xyz/cif files for the forms that aren't Wulff forms

### Definition of classes that load cif files, extract the data, and call the crystalNPs class to create ellipsoids, parallepipeds and spheres:

- Crystals_ellipsoids_parallepipeds
- Crystals_spheres
- Crystals_wires 

##### NB: Ellispoids and parallepipeds are in the same class because they can both be defined with a 3 elements array [d1,d2,d3],  d1, d2, d3 being the diameters in the 3 dimensions. Spheres are defined in their own class, they're described with a 1 element array [d], d being the diameter.

### Definition of Crystals_ellipsoids_parallepipeds

In [3]:
class Crystals_ellipsoids_parallepipeds :
    """
    A class for generating XYZ and CIF files of ellipsoidal and parallepipedic NPs 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data,sizes,forms,max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data,  shapes and sizes informations.
            Args:
        path (str) : path that will contain the xyz created files
        cif data (dataframe):  the cif files of different compounds
        sizes (array-like): array  of the 3 diameters of the ellipsoidal and parallepipedic NPs
        forms (array-like): array of the forms :  [ellipsoids, parallepipeds] or [ellipsoids] or [parallepipeds]
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): if bool=False : details of the files
            Methods:
        create_shapes(noOutput, max_size, path):  Generate ellipsoids and parallepipeds and their files for all the CIF compounds dataset.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # stock loaded cif files
        self.sizes= sizes
        self.forms= forms
        self.create_shapes(noOutput,max_size,forms,path)
        # number_created_files=0


    
    def create_shapes(self,noOutput,max_size,forms,path):
        """
        Generate ellipsoids and parallepipeds and their files for all the CIF compounds dataset.
            Args:
        noOutput (bool): if bool=False : details of the files
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        forms (array-like): array of the forms :  [ellipsoids, parallepipeds] or [ellipsoids] or [parallepipeds]
        path (str) : path that will contain the xyz created files
        
        """
        
        for cif_name, cif_file in self.cif_data['cif file'].items():
            # load cif informations
            self.cif_name=cif_name
            if len(self.cif_name.split())==2 : #for the name of the files, for example 'Rutile' or 'Anatase'
                structure=self.cif_name.split()[1]
            else : 
                structure=None
            if not noOutput :
                print(f'\n\033[1m{bg.LIGHTBLUEB}{cif_name.center(50)}\033[0m\n')
            cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
            crystal_system_name = self.ucBL.__class__.__name__
            direction=[0,0,1]
            d_hkl=pyNMBu.interPlanarSpacing(direction,self.ucUnitcell,crystal_system_name)*0.1 #nm    
            if not noOutput:
                print(f'\033[1m d_hkl= {d_hkl}nm')


            
            if not self.forms == None : 
                for form in self.forms :
                    if not noOutput:
                        print(f'\n\033[1m{bg.LIGHTBLUEB}{form}\033[0m\n')
                    index=0 
                    # instance of the crystal class for each cif files and forms
                    if form=='parallepiped' or form=='ellipsoid':
                        for i in self.sizes :
                            size= [i[0]*d_hkl,i[1]*d_hkl,i[2]*d_hkl]
                            index += 1 
                            TestNP = cyNP.Crystal(
                                crystal=f'{cif_name}',
                                userDefCif=cif_info['cif_path'],
                                shape=f" {form}",
                                size= size ,
                                buildPPD='abc',
                                threshold=0.001,
                                thresholdCoreSurface=2,
                                postAnalyzis=True,
                                jmolCrystalShape=True,
                                noOutput=True,
                                aseView=False,
                                skipSymmetryAnalyzis=True
                            ) 
                            circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                            if circumsphere_diameter<max_size :
                                if not noOutput :
                                    print(f'\033[1m Generating size is {size} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                                    print(f'\033[1m  Circumscribed sphere diameter ={circumsphere_diameter} nm \033[0m')
                                pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)
                            else :
                                if not noOutput :
                                    print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm \033[0m') 
                                break

                    
                    else: 
                        if not noOutput :
                            print(f"{bg.LIGHTREDB} Shape {form} is not an ellipsoid nor a parallepiped.\033[0m  ")        
       
            else: 
                if not noOutput :
                    print(f"{bg.LIGHTREDB} Please give an array of the forms : ['ellipsoid','parallepiped'] or ['parallepiped'] or ['ellipsoid'].\033[0m  ")        
                       
    

#### Example of use of the class Crystals_ellipsoids_parallepipeds 


In [4]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/ellips_para_1_5nm'

# Sizes of the NPs
size = [[2, 2, d] for d in np.arange(2, 25,1)] # d is a multiplier coefficient of dhkl, so the final sizes are : [2,2,2*dhkl], [2,2,3*dhkl], etc

# Instance of the class
# Possiblity to define a maximal size: max_size (circumsphere diameter). If max_size isn't defined, the NPs will have the size given (by multiple of dhkl).
class_test=Crystals_ellipsoids_parallepipeds(path,cif_data=data.pyNMBcif.CIFdf,sizes=size,max_size=5,forms=['ellipsoid','parallepiped'],noOutput=True)


print(t.chrono_stop(hdelay=True))

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'tetragonal' is not interpreted for space group Spacegroup(141, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


00:00:33 626ms


### Definition of Crystals_spheres

In [2]:
class Crystals_spheres :
    """
    A class for generating XYZ and CIF files of spherical NPs 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data,sizes,max_size,noOutput:bool = True):
        """
        Initialize the class with CIF data,  shapes and sizes informations.
            Args:
        path (str) : path that will contain the xyz created files
        cif data (dataframe):  the cif files of different compounds
        sizes (array-like): 1 element array of the diameter of the spheric NPs
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): if bool=False : details of the files
          Methods:
        create_shapes(noOutput, path):  Generate spheres and their files for all the CIF compounds dataset.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # stock loaded cif files
        self.sizes= sizes
        self.create_shapes(noOutput,max_size,path)
        # number_created_files=0


    
    def create_shapes(self,noOutput,max_size,path):
        """
        Generate spheres and their files for all the CIF compounds dataset.
        """
        
        for cif_name, cif_file in self.cif_data['cif file'].items():
            # load cif informations
            self.cif_name=cif_name
            if not noOutput :
                print(f'\n\033[1m{bg.LIGHTBLUEB}{cif_name.center(50)}\033[0m\n')
            if len(self.cif_name.split())==2 : #for the name of the files, for example 'Rutile' or 'Anatase'
                structure=self.cif_name.split()[1]
            else : 
                structure=None

 
            cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
            crystal_system_name = self.ucBL.__class__.__name__
            direction=[0,0,1]
            d_hkl=pyNMBu.interPlanarSpacing(direction,self.ucUnitcell,crystal_system_name)*0.1 #nm    
            if not noOutput:
                print(f'\033[1m d_hkl={d_hkl} nm \033[0m')

            index=0 
            for i in self.sizes :
                size= [i[0]*d_hkl]
                index += 1 
                TestNP = cyNP.Crystal(
                    crystal=f'{cif_name}',
                    userDefCif=cif_info['cif_path'],
                    shape='sphere',
                    size= size,
                    threshold=0.001,
                    thresholdCoreSurface=2,
                    postAnalyzis=True,
                    jmolCrystalShape=True,
                    noOutput=True,
                    aseView=False,
                    skipSymmetryAnalyzis=True
                ) 
                circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                if circumsphere_diameter<max_size :
                    if not noOutput :
                        print(f'\033[1m Generating size is {size} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                        print(f'\033[1m  Circumscribed sphere diameter ={circumsphere_diameter} nm \033[0m')
                    pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)     
                else :
                    if not noOutput :
                        print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm \033[0m') 
                    
                    break

                   
                
                       

#### Example of use of the class Crystals_spheres

In [3]:
t = pyNMBu.timer()
t.chrono_start()
size = [[d] for d in np.arange(2, 30, 1)]
print('size=',size)

path='database_1_5nm/spheres_1_5nm'

class_test=Crystals_spheres(path,cif_data=data.pyNMBcif.CIFdf,sizes=size,max_size=5,noOutput=False)


print(t.chrono_stop(hdelay=True))

size= [[2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17], [18], [19], [20], [21], [22], [23], [24], [25], [26], [27], [28], [29]]

                       NaCl                       

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1000041-NaCl.cif
 d_hkl=0.562 nm 
 Generating size is [1.124] nm and is equal to dhkl multiplied by [2]. 
  Circumscribed sphere diameter =0.8430000000000004 nm 
xyz file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000001_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000001_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000001_0000000.script
 Generating size is [1.6860000000000002] nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =1.405 nm 
xyz file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000002_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000002_0000000.cif
scr

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


 Generating size is [2.248] nm and is equal to dhkl multiplied by [4]. 
  Circumscribed sphere diameter =1.9670000000000003 nm 
xyz file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000003_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000003_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000003_0000000.script
 Generating size is [2.8100000000000005] nm and is equal to dhkl multiplied by [5]. 
  Circumscribed sphere diameter =2.529 nm 
xyz file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000004_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000004_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000004_0000000.script
 Generating size is [3.3720000000000003] nm and is equal to dhkl multiplied by [6]. 
  Circumscribed sphere diameter =3.091000000000001 nm 
xyz file created: database_1_5nm/spheres_1_5nm/NaCl_fcc_sphere_0000005_0000000.xyz
cif file cr

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'tetragonal' is not interpreted for space group Spacegroup(141, setting=1). This may result in wrong setting!
  warnings.warn(


 Generating size is [2.85429] nm and is equal to dhkl multiplied by [3]. 
  Circumscribed sphere diameter =2.838375 nm 
xyz file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000002_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000002_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000002_0000000.script
 Generating size is [3.8057200000000004] nm and is equal to dhkl multiplied by [4]. 
  Circumscribed sphere diameter =3.6077654742000007 nm 
xyz file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000003_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000003_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000003_0000000.script
 Generating size is [4.75715] nm and is equal to dhkl multiplied by [5]. 
  Circumscribed sphere diameter =4.730625 nm 
xyz file created: database_1_5nm/spheres_1_5nm/TiO2_anatase_sphere_0000004_0000000.xy

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


 Generating size is [1.71648] nm and is equal to dhkl multiplied by [6]. 
  Circumscribed sphere diameter =1.5734400000000004 nm 
xyz file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000005_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000005_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000005_0000000.script
 Generating size is [2.00256] nm and is equal to dhkl multiplied by [7]. 
  Circumscribed sphere diameter =1.8595199999999998 nm 
xyz file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000006_0000000.xyz
cif file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000006_0000000.cif
script file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000006_0000000.script
 Generating size is [2.28864] nm and is equal to dhkl multiplied by [8]. 
  Circumscribed sphere diameter =2.1455999999999995 nm 
xyz file created: database_1_5nm/spheres_1_5nm/Fe_bcc_sphere_0000007_0000000.xyz
cif file created: database_1_5n

### Definition of Crystals_wires

In [ ]:
class Crystals_wires :
    """
    A class for generating XYZ and CIF files of nanowires from a dataset of compounds (CIF dataset).
    This process enables the creation of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.

    """
    
    def __init__(self, path,cif_data,sizes,max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data, Wulff shapes information and size.
            Args:
        path (str) : path that will contain the xyz created files
        cif data (dataframe):  the cif files of different compounds
        wulff_shapes (dataframe): the wulff shapes and their informations
        sizes (array-like): array of the diameter and the legnth of the nanowires
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): if bool=False : details of the files
            Notes:
        directionWire (array-like): direction of the wires: [0,0,1] for hcp and [1,1,1] for cubic
        RotWire (int): number of edges of the wire's cross section: 4 and 6
        refPlaneWire (array-like): plane that is multiplied to create the wire: [1,0,0] for hcp and [0,1,-1] for cubic
            Methods:
        create_shapes(noOutput, path):  Generate wires and their files for all the CIF compounds dataset.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # stock loaded cif files
        self.sizes= sizes
        self.create_shapes(noOutput,max_size,path)
        
    def create_shapes(self,noOutput,max_size,path):
        """
        Generate spheres and their files for all the CIF compounds dataset.
        """
        import math
        for cif_name, cif_file in self.cif_data['cif file'].items():
            # load cif informations
            self.cif_name=cif_name
            if not noOutput :
                print(f'\n\033[1m {bg.LIGHTBLUEB} {cif_name.center(50)}\033[0m\n')
                
            if len(self.cif_name.split())==2 : #for the name of the files, for example 'Rutile' or 'Anatase'
                structure=self.cif_name.split()[1]
            else : 
                structure=None
            
            cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
           
            crystal_system_name = self.ucBL.__class__.__name__
            print('crystal_system_name',crystal_system_name)
            # instance of the crystal class for each cif files 
            if self.crystal_type =='fcc' or  self.crystal_type=='bcc' or self.crystal_type=='cubic':
                directionWire=[1,1,1]
                refPlaneWire=[0,1,-1]
            if  self.crystal_type =='hcp' :    
                directionWire=[0,0,1]
                refPlaneWire=[1,0,0]
            if not noOutput :
                print(f'refPlaneWire={refPlaneWire}and directionWire={directionWire}\n')

            
            if  self.crystal_type =='fcc':
                d_hkl_diameter=pyNMBu.interPlanarSpacing(refPlaneWire,self.ucUnitcell,crystal_system_name)*0.1 #nm
                d_hkl_length=2*pyNMBu.interPlanarSpacing(directionWire,self.ucUnitcell,crystal_system_name)*0.1 #nm   #multiply by otherwise same sizes twice
                print(f'\033[1m 2*d_hkl_length is {d_hkl_length} nm\033[0m')
                print(f'\033[1m d_hkl_diameter is {d_hkl_diameter} nm\033[0m')
            else:
                d_hkl_length=pyNMBu.interPlanarSpacing(directionWire,self.ucUnitcell,crystal_system_name)*0.1 #nm
                d_hkl_diameter=pyNMBu.interPlanarSpacing(refPlaneWire,self.ucUnitcell,crystal_system_name)*0.1 #nm
            
                print(f'\033[1m d_hkl_length is {d_hkl_length} nm\033[0m')
                print(f'\033[1m d_hkl_diameter is {d_hkl_diameter} nm\033[0m')

            
            for nRot in [4,6] :
                print(f'\n\033[1m{bg.LIGHTBLUEB} nRot (cross section of the wire) ={nRot}\033[0m\n')
                index=0
                for i in self.sizes :
                    size= [i[0]*d_hkl_diameter,i[1]*d_hkl_length]
                    index += 1 
                    TestNP = cyNP.Crystal(
                        crystal=f'{cif_name}',
                        userDefCif=cif_info['cif_path'],
                        shape='wire',
                        size= size,
                        directionWire= directionWire,
                        nRotWire= nRot,
                        refPlaneWire= refPlaneWire,
                        threshold=0.001,
                        thresholdCoreSurface=1,
                        postAnalyzis=True,
                        jmolCrystalShape=True,
                        noOutput=True,
                        aseView=False,
                        skipSymmetryAnalyzis=True
                    ) 
                       
                    circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                    if circumsphere_diameter<max_size :
                        if not noOutput :
                            print(f'\033[1m Generating size is {size} nm and is equal to dhkl multiplied by {i}.\033[0m ')
                            print(f'\033[1m  Circumscribed sphere diameter ={circumsphere_diameter} nm \033[0m')
                        pyNMBu.writexyz_generalized_crystals(self,structure,path,TestNP,index,noOutput)     
                    else :
                        if not noOutput :
                            print(f'\033[1m Circumscribed sphere diameter ={circumsphere_diameter} nm is greater than the maximum diameter defined={max_size} nm \033[0m') 
                        
                        break    
                    
                    
                    
                    
      
         

#### Example of use of the class Crystals_wires


In [ ]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/wires_1_5nm'

# Sizes of the NPs
size = [[2, d] for d in np.arange(2, 30, 1)] # d is a multiplier coefficient of dhkl, so the final sizes are : [2,2*dhkl], [2,3*dhkl], ect.

# Instance of the class
# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the size given (by multiple of dhkl).
class_test=Crystals_wires(path,cif_data=data.pyNMBcif.CIFdf,sizes=size,max_size=5,noOutput=False) # Here a maximum of a 5 nm for the circumscribed sphere diameter.


print(t.chrono_stop(hdelay=True))

# Create I=f(q) files from all the xyz files created

- qmin=0.001, qmax=20 and qstep=0.001 A
- time (CPU): it took approximatively 16 hours for 66 files, for the whole database = 4428, it would take 44 days ... 

In [ ]:
pyNMBuDC.create_iqfiles_from_xyzfiles?

In [13]:
t = pyNMBu.timer()
t.chrono_start()


path_of_coords='database_1_5nm/all_forms_1_5nm_xyz'
path_of_csvfiles=o'Iq_1_5nm_csv'
calc = DebyeCalculator(qmin=0.001, qmax=20.0, qstep=0.001, qdamp=0.0, biso=0.0)
pyNMBuDC.create_iqfiles_from_xyzfiles(calc,path_of_coords,path_of_csvfiles,noOutput=True)


print(t.chrono_stop(hdelay=True))

/tmp/ipykernel_195776/2201458071.py:3: UserWarning: Warning: Your system might have a CUDA-enabled GPU, but CUDA is not available. Computations will run on the CPU instead. For optimal performance, please install Pytorch with CUDA support. If you do not have a CUDA-enabled GPU, you can surpress this warning by specifying the 'device' argument as 'cpu'
  calc = DebyeCalculator(qmin=0.001, qmax=20.0, qstep=0.001, qdamp=0.0, biso=0.0)


KeyboardInterrupt: 